# AI Tokenomics — measuring a prompt instead of guessing at it

Code Europe, 15 September 2026 · Grzegorz Wasilewski (Accenture)

This notebook reproduces the cost analysis from the talk. It runs on a free
Colab runtime, with **no API key and no billing account**, by serving a local
model through Ollama.

The thing worth paying attention to: every other cost notebook you will find
estimates tokens with a tokenizer and then *assumes* what the model will
answer. This one asks the model and reads its own counters, then shows you
how far apart those two answers are.

### What gets measured, and what does not

| claim | how it is established |
|---|---|
| input tokens per call | measured — the model reports `prompt_eval_count` |
| output tokens per call | measured — the model reports `eval_count` |
| accuracy against ground truth | measured — parsed ids compared to a hand-labelled set |
| prompt truncation | measured — the counter stops at the context ceiling |
| **dollar cost** | **calculated** — measured tokens × published list prices |
| **cache savings** | **calculated** — a local model does not bill, so there is no cache counter to read |

The last two rows are arithmetic on top of real token counts, not
observations. Said plainly so nobody has to guess which is which.

---

### This is demonstration code, not production code

Everything here exists to make a cost argument visible and reproducible on a
free Colab runtime. It is **not** written to be deployed:

* There is no error handling worth the name, no retry policy, no rate
  limiting, no authentication, no input validation and no secrets management.
* The classification prompts are built to be *measurable*, not to be good.
  Accuracy here tops out around 73% — that is a property of this setup, not a
  recommendation.
* The taxonomy, the feedback corpus and the ground-truth labels are
  **synthetic**. They are modelled on a real incident; they are not its data.
* Prices are list prices at a point in time and will be stale.

Read it as "here is one way this could be implemented, and here is what it
costs", not as something to lift into a system.

## 1. Runtime setup

Skipped automatically when you are not on Colab.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
print(f"Colab runtime: {IN_COLAB}")

### What machine did you actually get?

Colab's documentation says plainly that the machine types "vary over time", so
the only trustworthy answer comes from the runtime itself. Run this before
committing to a long benchmark — it takes a second and tells you whether the
model you want will fit.

### Picking an accelerator

**Skip the TPUs.** Ollama's hardware documentation has exactly four sections —
Nvidia, AMD Radeon, Metal, Vulkan — and llama.cpp's backend table has no TPU
entry at all. The reason is structural: code for a TPU has to be compiled by
the XLA compiler, and llama.cpp is hand-written CUDA and Metal kernels.
Selecting a TPU runtime gives you a **CPU-only** Ollama.

Among the GPUs, note what this workload actually is. Prefill of a
14,159-token prompt costs roughly **340 TFLOP**, while the weights are read
once. It is **compute-bound, not bandwidth-bound** — which reverses the
intuition you may have: L4 has *lower* memory bandwidth than T4 (300 vs
320 GB/s) and is still much faster here, because dense FP16 tensor throughput
is what gets used.

| GPU | VRAM | dense FP16 | max ctx for `gemma3:12b` | fits a 14.2k prompt? |
|---|---|---|---|---|
| T4 | 16 GB | 65 TFLOPS | ~16,300 | yes, 13% headroom |
| L4 | 24 GB | 121 TFLOPS | ~36,800 | comfortably |
| A100 | 40 GB | 312 TFLOPS | ~83,200 | comfortably |

On spec sheets alone the A100 is about **4.8x** a T4 and **2.6x** an L4 for
prefill. That is a **ratio of published TFLOPS, not a benchmark** — real gains
are lower, because 4-bit dequantization and kernel efficiency differ by
architecture. Treat it as a ceiling.

The TFLOPS above are **dense** figures. NVIDIA's own pages headline the
*sparsity* numbers, which are exactly twice as large — the L4 page prints 242
and the A100 datasheet prints "312 | 624\*". If you cross-check and the numbers
look half-size, that is deliberate: sparsity does not apply to this workload.

The context column is **calculated**: `KV = 2 x layers x kv_heads x head_dim x
ctx x 2` (FP16), less weights, less ~1.5 GB reserve. It is also a conservative
**upper bound for Gemma 3 specifically**, which interleaves 5 sliding-window
layers per global layer — with that accounted for, KV at 14,159 tokens is
closer to 1.2 GiB than the 5.2 GiB the dense formula predicts. Neither number
is measured. The cell above is.

In [ ]:
def machine_report() -> None:
    try:
        gpu = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True,
        ).stdout.strip()
    except FileNotFoundError:
        gpu = ""          # no nvidia-smi at all: CPU runtime, a TPU, or a Mac
    print("accelerator :", gpu or "no NVIDIA GPU visible (CPU-only, TPU runtime, or Apple Silicon)")

    try:
        with open("/proc/meminfo") as fh:
            total = next(l for l in fh if l.startswith("MemTotal:"))
        print(f"host RAM    : {int(total.split()[1]) / 1024**2:.1f} GB")
    except FileNotFoundError:
        print(f"host RAM    : {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3:.1f} GB")

    print("vCPU        :", os.cpu_count())
    disk = shutil.disk_usage("/content" if IN_COLAB else ".")
    print(f"disk        : {disk.free / 1024**3:.0f} GB free of {disk.total / 1024**3:.0f} GB")


machine_report()

In [ ]:
# Ollama needs these to detect the GPU; a bare Colab image has none of them.
if IN_COLAB:
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq pciutils lshw zstd",
        shell=True, check=True,
    )
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

    # This sequence — the three apt prerequisites, the installer, then serve in a
    # Popen with a readiness loop — is adapted from
    # github.com/Legard777/eskadra-bielik-misja2 (Apache-2.0), itself a fork of
    # avedave's work. Kept here as well as in NOTICE because the notebook is the
    # file people actually open, and an attribution nobody reads is decoration.

In [ ]:
# `ollama serve` has to outlive the cell, so it goes into a Popen, not a `!`.
OLLAMA_HOST = "http://localhost:11434"



OLLAMA_LOG = pathlib.Path("/tmp/ollama-serve.log")


def ollama_log_tail(n: int = 14) -> str:
    """The last few lines the server wrote. Empty string if there is nothing."""
    try:
        return "\n".join(OLLAMA_LOG.read_text(errors="replace").splitlines()[-n:])
    except Exception:
        return ""


def ollama_up(host: str = OLLAMA_HOST, timeout: float = 2.0) -> bool:
    import urllib.request
    try:
        urllib.request.urlopen(f"{host}/api/tags", timeout=timeout).read()
        return True
    except Exception:
        return False


def ensure_ollama(host: str = OLLAMA_HOST) -> None:
    """Start the server if it is not already answering. Safe to call repeatedly.

    Notebooks get run out of order, and a Colab runtime can restart underneath
    you and take the server process with it. Anything that needs the model calls
    this first, so no cell depends on another cell having been run.
    """
    if ollama_up(host):
        return

    print("Ollama server not found or not responding. Attempting to restart...")
    if shutil.which("ollama") is None:
        raise SystemExit("ollama is not installed — run the install cell above")

    # Keep the log. It used to go to DEVNULL, which is why an HTTP 500 from a
    # model load looked like an act of God: Ollama runs a SEPARATE runner
    # process per model, and when the kernel OOM-kills that runner the parent
    # survives, returns 500, and starts a new one. From outside it looks like
    # the server restarted itself. The reason is in this file and nowhere else.
    OLLAMA_LOG.parent.mkdir(parents=True, exist_ok=True)
    log = open(OLLAMA_LOG, "ab", buffering=0)
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)
    for _ in range(60):          # the port opens before the server answers
        if ollama_up(host):
            print("Ollama server is now running.")
            return
        time.sleep(1)
    raise SystemExit("Ollama server did not come up within 60s after restart attempt.")


def unload_model(name: str, host: str = OLLAMA_HOST) -> bool:
    """Drop a model out of memory now, without deleting it.

    This is the fix for the matrix loop. Ollama keeps a model resident for five
    minutes after the last request, so a loop over four models holds two or
    three of them at once and the next load fails with HTTP 500 on a 16GB card.

    `keep_alive: 0` is Ollama's own mechanism for this: an empty request that
    tells the server to unload the model immediately. It frees VRAM only —
    **the model stays pulled on disk**, so nothing has to be downloaded again.
    That is the difference between this and `ollama rm`, and it is the reason
    this notebook does the former and never the latter: re-pulling gigabytes on
    conference wifi is not a recovery plan.
    """
    import json as _json, urllib.request
    body = _json.dumps({"model": name, "keep_alive": 0}).encode()
    req = urllib.request.Request(f"{host}/api/generate", data=body,
                                 headers={"Content-Type": "application/json"})
    try:
        urllib.request.urlopen(req, timeout=30).read()
        return True
    except Exception as exc:
        print(f"    could not unload {name}: {exc}")
        return False


def restart_ollama(host: str = OLLAMA_HOST) -> None:
    """Stop the server and bring it back up.

    Needed because `ensure_ollama()` deliberately does nothing when the server
    answers — and the failure this handles is one where it DOES answer. Ollama
    keeps a model resident after a run; on a 16GB T4 the third or fourth model
    of a matrix can then fail to load with an HTTP 500 while `/api/tags` stays
    perfectly healthy. From the outside the server looks fine. It is out of
    VRAM.

    Restarting drops whatever is resident. It does **not** remove anything:
    every model stays pulled on disk and the next run loads it again. Deleting
    models between runs would also free the memory, and would mean re-pulling
    gigabytes on a conference wifi — so this does not do that.
    """
    subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    time.sleep(2)
    ensure_ollama(host)


ensure_ollama()
print("ollama is answering on", OLLAMA_HOST)

### Choosing a model

A free Colab runtime is **not guaranteed a GPU**. Google states plainly that
"the types of GPUs and TPUs that are available in Colab vary over time", so
there is no documented tier-to-card mapping. In practice the free tier gives a
**T4 with 16 GB**, and everything below is sized for that.

The trap: **context costs memory separately from weights, and it is usually
context that runs out first.** A model advertising a 128K window will not give
you 128K on a T4, because the KV cache does not fit.

| Ollama tag | Params | Download | Declared ctx | **Real max ctx @16 GB** |
|---|---|---|---|---|
| `llama3.2:3b` | 3.2B | 2.0 GB | 131,072 | ~98k |
| `phi4-mini:3.8b` | 3.8B | 2.5 GB | 131,072 | ~82k |
| `gemma3:4b` | 4.3B | 3.3 GB | 131,072 | ~71k |
| `llama3.1:8b` | 8.0B | 4.9 GB | 131,072 | ~64k |
| `deepseek-r1:8b` | 8.2B | 5.2 GB | 131,072 | ~55k |
| `qwen3:8b` | 8.2B | 5.2 GB | 40,960 | **40,960 — all of it** |
| `mistral:7b` | 7.2B | 4.4 GB | 32,768 | **32,768 — all of it** |
| `phi4:14b` | 14.7B | 9.1 GB | 16,384 | 16,384 — all of it |
| `mistral-nemo:12b` | 12.2B | 7.1 GB | **1,024,000** | ~38k |
| `qwen3:14b` | 14.8B | 9.3 GB | 40,960 | ~25k |
| `deepseek-r1:14b` | 14.8B | 9.0 GB | 131,072 | ~22k |
| `gemma3:12b` | 12.2B | 8.2 GB | 131,072 | ~13-16k dense, more with SWA |
| `gemma4:12b` | 11.9B | 7.0 GB | 262,144 | **9.2 GB resident at 64k — measured** |

Every tag above was checked against the Ollama registry and exists. The
**"real max ctx" column is calculated, not measured**, and two independent
calculations of the `gemma3:12b` row landed at ~13k and ~16.3k depending on how
much VRAM was reserved — which is a fair signal of how much these numbers are
worth. Gemma 3's sliding-window layers push the real figure higher still.
Formula:
`KV_bytes = 2 x layers x kv_heads x head_dim x context x 2` (FP16), against a
15 GB budget less weights less 1.5 GB for activations. Layer counts and head
dimensions were read out of the GGUF headers Ollama actually downloads, not
from model cards. Quantizing the KV cache to `q8_0` roughly halves these
numbers; the two Gemma rows ignore its sliding-window saving, so they are
conservative floors.

Look at `mistral-nemo:12b`. It declares a **1,024,000-token** context. Serving
that would need roughly **168 GB** of KV cache. The number in the model card
is real; it is just not a number about your machine.

The `gemma4:12b` row is the exception: it is **measured, not calculated**. Its
GGUF header leaves `attention.head_count_kv` empty and splits `key_length`
between global (512) and sliding-window (256) layers, so the usual formula
cannot be applied at all. Loading it and reading `ollama ps` gave **9.2 GB
resident with a 65,536-token window, 100% on GPU** — which is the number that
actually decides whether it fits, and it fits a 16 GB T4 with ~6 GB to spare.
When the arithmetic is not available, measure instead of guessing.

**Pick `qwen3:8b` if it simply has to start** — weights plus a full-context KV
cache come to about 12.8 GB, and it is the only model here that runs its
entire declared window unmodified. Drop to `llama3.2:3b` or `gemma3:4b` on a
bad T4 day. Above 32k, `llama3.1:8b` has the best quality-to-context ratio.
Avoid the 14B class above 32k entirely.

One caveat, because it is worth being precise about: `num_ctx` governs how
much KV cache gets allocated, and on a 16 GB card that is what decides whether
you fit. It does **not** reliably truncate a long prompt — measured here on
13 Sept 2026, a 1,173-token prompt sent with `num_ctx=512` was still evaluated
in full. Set it for memory planning, not as a safety limit.

In [ ]:
# ---------------------------------------------------------------------------
# SET THIS. It is the one choice that changes what the rest of the notebook
# measures, and a mismatch here invalidates any cross-machine comparison.
# ---------------------------------------------------------------------------
MODEL = "gemma3:4b"
#   "gemma3:4b"    safe anywhere, including CPU-only. Noticeably less accurate.
#   "gemma3:12b"   the reference model for the baseline in this repo. Needs a GPU.
#   "gemma4:12b"   newer, same tokenizer, measured at 9.2 GB resident — fits a T4.
#   "gemma4:31b"   real hardware only.
#
# An environment variable wins if set, so the same file runs unattended:
MODEL = os.environ.get("TOKENOMICS_MODEL", MODEL)
print("model:", MODEL)

def pull(model: str) -> None:
    """Pull, and on failure say WHY — not just 'exit status 1'."""
    ensure_ollama()          # `ollama pull` fails outright if nothing is serving
    print(f"Pulling model: {model}")
    # capture_output, not check=True: a bare CalledProcessError throws away the
    # one line that says what actually went wrong.
    r = subprocess.run(["ollama", "pull", model], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"{model} ready")
        return

    # ollama draws a progress spinner, so its output is full of terminal escapes
    import re
    said = re.sub(r"\x1b\[[0-9;?]*[a-zA-Z]|\r", "", r.stderr or r.stdout or "")
    lines = [l.strip() for l in said.splitlines() if l.strip()]
    # the real cause is on an "Error:" line; everything else is progress spinner
    errors = [l for l in lines if l.startswith("Error")]
    said = "\n".join(errors or lines) or "(nothing on either stream)"

    print(f"ollama pull {model} failed with exit {r.returncode}\n")
    print("--- ollama said ---")
    print(said.strip()[-2000:])

    for path in ("/", "/content"):
        try:
            u = shutil.disk_usage(path)
            print(f"\ndisk {path:9s} {u.free / 1024**3:6.1f} GB free of {u.total / 1024**3:.0f} GB")
        except OSError:
            pass
    print(f"\nserver answering: {ollama_up()}")
    print("Most common causes: the runtime restarted and took the server with it,")
    print("or the tag does not exist. Disk is rarely it — on Colab / and /content")
    print("are the same volume.")
    raise SystemExit(f"could not pull {model}")


if IN_COLAB or shutil.which("ollama"):
    pull(MODEL)

In [ ]:
# The measurement modules. On Colab these come from the public repo; locally
# they are already next to this notebook.
# Three ways to get the measurement modules, tried in order.
REPO_URL = "https://github.com/acn-codeeurope/ai-tokenomics.git"
REPO_DIR = "/content/ai-tokenomics"
UPLOAD_ZIP = "/content/notebook-modules.zip"   # fallback: upload it in the file panel


def _module_dir(root: str) -> str:
    """Find the modules rather than assume where they are.

    This used to hardcode f"{root}/notebook", which is true only if the repo
    keeps them in a subdirectory. Published flat — which is how this one is —
    the clone succeeds, the path is wrong, and the failure surfaces three cells
    later as `ImportError: no module named feedback`. On stage that reads as
    "his repo is broken", when the repo is fine and the assumption was not.
    """
    for cand in (root, f"{root}/notebook"):
        if os.path.exists(f"{cand}/feedback.py"):
            return cand
    raise SystemExit(
        f"cloned {root} but found no feedback.py in it or in {root}/notebook — "
        "the repository layout is not what this cell expects")


if not IN_COLAB:
    MODULE_DIR = os.getcwd()
elif os.path.isdir(REPO_DIR):
    MODULE_DIR = _module_dir(REPO_DIR)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    MODULE_DIR = _module_dir(REPO_DIR)
elif os.path.exists(UPLOAD_ZIP):
    import zipfile
    MODULE_DIR = "/content/notebook"
    with zipfile.ZipFile(UPLOAD_ZIP) as z:
        z.extractall(MODULE_DIR)
    print(f"extracted {UPLOAD_ZIP} -> {MODULE_DIR}")
else:
    raise SystemExit(
        "No modules available. Either set REPO_URL above, or upload "
        f"notebook-modules.zip to {os.path.dirname(UPLOAD_ZIP)} using the file "
        "panel on the left, then re-run this cell."
    )
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)

import feedback
import measure
import prompts
import rulebook
import runner

runner.DEFAULT_MODEL = MODEL
print("modules loaded from", MODULE_DIR)

## 2. The workload

The setup is the one from the incident in the talk: agents read customer
feedback and return a **list** of categories. In the incident the taxonomy
came from the business — **50 categories**, each market agreed to use only
**2 to 5** of them. The taxonomy you are about to see has the same shape and
was written for this notebook; the real one is not here. See `rulebook.py`.

That "2 to 5" is the whole story. It is the difference between a prompt that
carries the rules a market actually needs and a prompt that carries all fifty.

In [ ]:
print(f"categories in the rulebook : {len(rulebook.all_categories())}")
print(f"markets with an agreed scope: {len(rulebook.MARKET_SCOPE)}")
sizes = sorted(len(v) for v in rulebook.MARKET_SCOPE.values())
print(f"categories per market      : {sizes[0]} to {sizes[-1]}")
print(f"feedback items with labels : {len(feedback.ITEMS)}")

In [ ]:
# One category's rule block — this is the unit that gets multiplied.
import textwrap

example = rulebook.all_categories()[0]
block = example.rules_text()

# Wrapped for reading only; the token count below is taken from the original.
for line in block.split("\n"):
    print(textwrap.fill(line, width=88, subsequent_indent="    ") if len(line) > 88 else line)

print(f"\n-> {measure.count(block, 'o200k_base'):,} tokens for ONE category")

In [ ]:
# Across all 50, so we know the spread rather than one sample.
lens = [measure.count(c.rules_text(), "o200k_base") for c in rulebook.all_categories()]
print(f"rule block: min {min(lens)}, max {max(lens)}, mean {sum(lens)/len(lens):.0f} tokens")
print(f"all 50 blocks together: {sum(lens):,} tokens of prompt, on every single call")

## 3. Three ways to send the same work

| name | what the prompt carries | calls per item |
|---|---|---|
| `incident` | all 50 categories | 1 |
| `scoped` | only the 2–5 this market agreed to | 1 |
| `fanout` | one category per call | 2–5 |

`incident` is the state the system drifted into. `scoped` is what the design
said. `fanout` is the shape people reach for when they hear "split it up".

In [ ]:
item = feedback.ITEMS[0]
print(f"item {item.iid} · market {item.market} · ground truth {item.truth}\n")

for name, calls_for in (
    ("incident", prompts.incident_calls),
    ("scoped", prompts.attempt1_calls),
    ("fanout", prompts.attempt2_calls),
):
    calls = calls_for(item, rulebook.scope_for)
    tok = sum(measure.count(full, "o200k_base") for _, full in calls)
    print(f"{name:9s} {len(calls)} call(s)  {tok:6,} input tokens (tiktoken estimate)")

## 4. The estimate, against the model's own counter

Everything above is `tiktoken` counting characters it was never asked to bill.
Now we send the same text to the model and read `prompt_eval_count`.

Expect a gap. A local model does not use OpenAI's `o200k_base`, and **you are
billed by the tokenizer of the model you actually call**. The size of the gap
is the useful part: if it is a few percent, tokenizer estimates are a
defensible planning tool. If it is large, they are theatre.

In [ ]:
_, scoped_prompt = prompts.attempt1_calls(item, rulebook.scope_for)[0]

est = measure.count(scoped_prompt, "o200k_base")
usage = runner.generate(scoped_prompt, model=MODEL)

rows = [("tiktoken o200k_base estimate", f"{est:,}"),
        (f"{MODEL} prompt_eval_count", f"{usage.tokens_in:,}"),
        ("ratio", f"{usage.tokens_in / est:.3f}")]
w = max(len(k) for k, _ in rows)          # model names vary in length; align to the data
for k, v in rows:
    print(f"{k:<{w}s} : {v}")

print(f"\nthe model answered {usage.tokens_out} tokens in {usage.ms/1000:.1f}s:")
print(usage.text.strip()[:400])

### What the cost model assumed the answer would be

The projection in the talk assumes the reply is the id list and nothing else —
about five tokens. Models do not do that unless you make them. They wrap the
answer in JSON, or a code fence, or an explanation of the JSON in the code
fence.

Output is priced several times higher than input, so an assumption that is
wrong by a factor of seven is not a rounding error.

In [ ]:
assumed = '["' + '", "'.join(item.truth) + '"]' if item.truth else ""
assumed_tok = measure.count(assumed, "o200k_base")
print(f"assumed output : {assumed_tok:4d} tokens   {assumed}")
print(f"actual output  : {usage.tokens_out:4d} tokens")
if assumed_tok:
    print(f"off by         : {usage.tokens_out / assumed_tok:.1f}x")

## 5. The context wall

A prompt longer than the model's context window is not rejected. It is cut to
fit, and `prompt_eval_count` then reports the ceiling — a number shaped
exactly like a measurement.

This is the incident's own failure mode in miniature: the call succeeds, a
number gets reported, and nothing anywhere says that input was thrown away.

### The wall is a setting, not a property of the model

Measured on a Colab T4 on 14 Sept 2026, running this exact notebook:
**all four models truncated the incident prompt to ~2,050 tokens** — gemma3:12b,
gemma4:12b, qwen3:8b and deepseek-r1:8b alike, from 5 GB of weights to 8 GB,
every one of them declaring a ceiling of 40,960 or more.

Identical truncation across models of different sizes rules out memory
pressure as the cause. What they share is the **environment's default context
window**, and on that runtime it is 2,048 tokens. One setting, applied to
everything, throwing away 85% of every prompt.

On a laptop with 48 GB the same notebook truncated nothing, because the
default there was large enough — which is exactly why this is easy to miss:
**it works on the machine you developed it on.**

**Check which build you have.** The same tag can change under you. The local
copy of `gemma3:12b` used while writing this declared an 8,192-token ceiling
and pinned a 13,570-token prompt at exactly 8,192 — no error, no warning
field. An `ollama pull` replaced it with a build declaring 131,072, and the
same prompt then fit. Both observations are real; only one of them is a
property of the model.

In [ ]:
PRICE_IN, PRICE_OUT = measure.PRICE_IN, measure.PRICE_OUT
limit = runner.model_context(MODEL)
_, incident_prompt = prompts.incident_calls(item, rulebook.scope_for)[0]
incident_est = measure.count(incident_prompt, "o200k_base")

rows = [(f"{MODEL} declared ceiling", f"{limit:,} tokens"),
        ("incident prompt", f"{incident_est:,} tokens (tiktoken estimate)"),
        ("fits under the ceiling", str(incident_est < limit))]
w = max(len(k) for k, _ in rows)
for k, v in rows:
    print(f"{k:<{w}s} : {v}")
print()

# `expect_tokens` is what makes truncation detectable at all: without an
# independent estimate to compare against, a truncated count is just a number.
u = runner.generate(incident_prompt, model=MODEL,
                    options={"num_predict": 32}, expect_tokens=incident_est)
rows = [("prompt_eval_count", f"{u.tokens_in:,}"),
        ("ratio to estimate", f"{u.tokens_in / incident_est:.3f}"),
        ("truncated", str(u.truncated))]
w = max(len(k) for k, _ in rows)
for k, v in rows:
    print(f"{k:<{w}s} : {v}")

if u.truncated:
    print(f"\n-> The request SUCCEEDED and dropped roughly "
          f"{1 - u.tokens_in / incident_est:.0%} of the rules on the way in.")
    print("   No error. No warning. Only a smaller number.")
else:
    print("\n-> Whole prompt evaluated on this runtime's default window.")

### The same prompt, with the window set explicitly

One parameter decides whether the model reads your rules or 15% of them. The
cost of the truncated call is *lower*, which is the trap: a monitoring
dashboard reports the broken run as the cheap one.

In [ ]:
for label, ctx in (("environment default", None), ("num_ctx set to 32,768", 32768)):
    opts = {"num_predict": 16}
    if ctx:
        opts["num_ctx"] = ctx
    v = runner.generate(incident_prompt, model=MODEL, options=opts,
                        expect_tokens=incident_est)
    usd = v.tokens_in * PRICE_IN / 1e6 * 1000
    print(f"{label:24s} in {v.tokens_in:7,} tokens  truncated={str(v.truncated):5s}  "
          f"${usd:6.2f} per 1000 runs")

print("\nThe cheaper row is the broken one.")

## 6. From tokens to money

Tokens above are **measured**. Dollars below are **calculated** — measured
tokens times published list prices. Nothing here observed a bill.

In [ ]:
import run_live

sample = feedback.ITEMS[:3]          # raise once you know how fast your runtime is
results = [run_live.run_architecture(n, sample, model=MODEL, progress=False)
           for n in ("scoped", "fanout")]
run_live.compare(results)

### Scaling up, and where the cache comes in

`crossover.py` projects the same three architectures across a month of volume
and applies the cache multipliers — cached reads at 0.1x input, cache writes
at 1.25x. Those multipliers are **list-price arithmetic, not measurements**: a
local model has no billing, so there is no cache counter to read.

Run it for the full table:

In [ ]:
import crossover

crossover.table()          # the projection, including the modelled cache columns

## 7. Did the cheap one get the right answer?

A cheaper architecture that is wrong is not cheaper. `feedback.ITEMS` carries
a hand-labelled category set per item, so the answers can be scored.

Watch precision separately from recall. A model that returns *every* plausible
category scores perfect recall and is still useless — it has moved the work of
deciding back onto whoever reads the output.

In [ ]:
# How many categories does it hand back, against how many are actually right?
# This is the number the precision column is really describing.
for r in results:
    predicted = sum(s["tp"] + s["fp"] for s in r.scores) / len(r.scores)
    truth = sum(s["tp"] + s["fn"] for s in r.scores) / len(r.scores)
    print(f"{r.name:9s} returns {predicted:.1f} categories per item, "
          f"{truth:.1f} are correct  ->  {predicted / truth:.1f}x over-production"
          f"   (recall {r.recall:.0%})")

## 8. The architecture from the slide, as running code

```
START -> fetch_rules -> narrow_scope -> classify_shard (xN) -> join_ids -> persist_raw
```

Five nodes. **Four of them never call a model.** The rules arrive from
storage, the scope comes from a field on the item, the join is set union, and
the write is issued by application code. Exactly one node spends tokens.

That is not a stylistic preference. A deterministic node can be unit-tested,
rate-limited and given a budget. A model call can only be asked nicely.

LangGraph is used here because it has a **named primitive for each move**:
`Send` for the runtime fan-out, an `operator.add` reducer for the join, and
`add_conditional_edges` for the branch. Hand-rolled, all three disappear into
a list comprehension and stop being visible in a review.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "langchain-ollama", "langgraph", "grandalf"], check=True)

import graph_app

app = graph_app.build(MODEL)
llm = graph_app.make_llm(MODEL)
try:
    drawing = app.get_graph().draw_ascii()       # needs grandalf
except Exception:
    drawing = app.get_graph().draw_mermaid()     # always available

# grandalf pads every line to a fixed width; strip it so the box is not
# surrounded by invisible whitespace.
print("\n".join(line.rstrip() for line in drawing.split("\n")))

In [ ]:
result = graph_app.classify(feedback.ITEMS[0], app=app, llm=llm)
print(f"ids returned  : {result['ids']}")
print(f"ground truth  : {result['truth']}")
print(f"llm calls     : {result['llm_calls']}   (one per in-scope category)")
print(f"input tokens  : {result['input_tokens']:,}")
print(f"output tokens : {result['output_tokens']:,}")
print(f"precision     : {result['score']['precision']:.0%}"
      f"   recall: {result['score']['recall']:.0%}")

### The node tally

| | nodes | model calls |
|---|---|---|
| deterministic logic | 4 | 0 |
| agent logic | 1 | N, one per in-scope category |

The counters above come from `ChatOllama`. It exposes `usage_metadata` with
`input_tokens` / `output_tokens`, and keeps Ollama's raw `prompt_eval_count`
and `eval_count` in `response_metadata` — verified, not assumed, so nothing
here is re-counted with a tokenizer.

## What this notebook established

**Measured**

* Input tokens per call, from the model's own counter.
* Output tokens per call — and how badly a cost model does when it assumes
  the answer is just the id list.
* Silent truncation: a prompt past the context window is cut, not refused,
  and the counter then reports the ceiling as if it were a measurement.
* Accuracy against a hand-labelled set, precision separately from recall.

**Calculated, not measured**

* Every dollar figure — measured tokens times published list prices.
* Every cache figure — a local model does not bill, so there is no cache
  counter to read. The 0.1x read and 1.25x write multipliers are list-price
  arithmetic applied on top of real token counts.

**Not established here**

* That any of this transfers to the billed model. A local model has a
  different tokenizer and different weights. What transfers is the *method*:
  count what you send, count what comes back, and never let a projection stand
  in for either one.

## 9. Same experiment, different machine

Everything above was first measured on a MacBook — an M4 Pro with 48 GB of
unified memory, running Ollama on Metal. A Colab T4 has 16 GB of dedicated
VRAM and a completely different memory model. Neither is the machine you would
bill in production, which is exactly why the comparison is worth making: it
shows which numbers are properties of the *workload* and which are properties
of the *hardware*.

The expectation, stated before running so it can be wrong:

* **Token counts and accuracy should be identical.** Same model, same prompts,
  `temperature=0`. If they differ, the two runs were not the same experiment.
* **Only the clock should change.**

One thing to watch on a T4: `gemma3:12b` has a `head_dim` of 256, so its KV
cache is unusually expensive — roughly 13k tokens of context inside 16 GB. The
incident prompt is about 14.1k tokens. If it does not fit in VRAM, Ollama will
fall back to system memory and the incident row will be *dramatically* slower
rather than failing outright. That is worth seeing once.

In [ ]:
import bench

import re

BENCH_ITEMS = 3          # the incident row is the slow one; raise once you know the cost

# The machine names the file. "bench-colab.json" from a T4 and from an A100 are
# two different experiments with one filename — that already cost one result
# here, overwritten and recovered from notes.
MACHINE = re.sub(r"[^a-z0-9]+", "-", bench.environment(MODEL)["label"].lower()).strip("-")
RESULT_PATH = f"bench-{MACHINE}.json"
print("machine:", MACHINE, "->", RESULT_PATH)

# A warmup call absorbs model-load time so it does not land on the first
# architecture and make it look slow.
# progress=True: a cell that prints nothing for fifteen minutes is a cell you
# cannot tell apart from a hung one.
this_run = bench.run(BENCH_ITEMS, model=MODEL, out_path=RESULT_PATH, progress=True)

In [ ]:
# Version-matched baseline. Colab ships a recent Ollama, so comparing against a
# run from an older one measures the software upgrade and calls it hardware —
# on 14k-token prompts that mistake is worth 10.8x.
BASELINE = os.path.join(MODULE_DIR, "results-macbook-ollama034.json")

runs = []
if os.path.exists(BASELINE):
    runs.append(bench.load(BASELINE))
else:
    print(f"no baseline at {BASELINE} — showing this machine only\n")
runs.append(this_run)

bench.compare(runs)

### Reading the comparison

`minutes` is not comparable when the two runs used a different number of
items — `sec/call` and `in tok/s` are. Input throughput is the honest
cross-machine number: it is tokens of prompt the hardware chewed per second,
independent of how many items you asked for.

If the token columns match and only the clock moved, the measurement is
portable and the cost conclusions hold anywhere. That is the result worth
having. A cost model that changes answer depending on whose laptop ran it is
not a cost model.

**How much difference counts as a difference.** Measured, not guessed: four
back-to-back runs of the identical workload on one machine landed between
207.8 and 211.0 input tokens/sec — a spread of **1.02x, about ±1%**. This
benchmark is repeatable.

What does move it is **competition for the same Ollama server**. An earlier
baseline here was collected while other jobs were hitting the same instance
and came in around 30% slower, which looked exactly like a hardware
difference and was not. Run the benchmark with nothing else talking to
Ollama, or you will be comparing schedulers rather than machines.

Token counts and accuracy are immune to this — they do not depend on load.
Only the clock columns are at risk.

## 10. The same workload across several models

Everything above runs one model. This section runs a set of them and puts the
results side by side, because the interesting question is not "what does this
model cost" but "what does the *choice* cost".

### The settings are not optional, and they were expensive to find

Each model below needs a specific configuration, discovered by measurement:

| model | setting | what happens without it |
|---|---|---|
| `gemma3:12b` | none | — |
| `gemma4:12b` | `think=False` | reasoning never terminates; it eats the entire output budget (measured at 256, 1024 *and* 4096 tokens) and returns an **empty** answer |
| `qwen3:8b` | `think=False` | same failure mode |
| `deepseek-r1:8b` | `think=False`, `num_predict≥1024` | needs ~497 output tokens to finish. At 256 it is cut off mid-answer |

A reasoning model with no stop token will spend everything you give it on
thinking you never see — and output is billed several times higher than
input. That is full price for no answer, with nothing in the logs.

`deepseek-r1:8b` has a second quirk worth knowing: it answers `["001"]` where
the prompt asked for `CAT-001`. The category is right and the contract is
broken — it would fail any caller expecting the documented id. `parse_ids()`
accepts it, and the `drift` column counts how often it happened.

In [ ]:
# Comment out what you do not want to wait for. Each model is roughly one to
# five minutes for 15 items, except deepseek-r1, which is slower because it
# generates an order of magnitude more output.
MODELS = {
    "gemma3:12b":     dict(think=None,  num_predict=None),
    "gemma4:12b":     dict(think=False, num_predict=None),
    "qwen3:8b":       dict(think=False, num_predict=None),
    "deepseek-r1:8b": dict(think=False, num_predict=1024),
}

MATRIX_ITEMS = 5          # raise to 15 for the full corpus

In [ ]:
import probe_model

for name in MODELS:
    pull(name)                       # no-op if the model is already present

# Same discipline as the matrix further down, and for the same reason.
#
# probe() makes four real calls, so it loads the model into VRAM exactly the way
# a benchmark does — and Ollama then holds it for five minutes. Probing four
# models without unloading fills a 16GB card BEFORE the matrix has measured a
# single item, which is the worse version of the problem: the run dies during
# the part that is supposed to be cheap.
#
# The unload sits in `finally`. A probe that fails still loaded the model, and
# leaving it resident is exactly how the next one fails too.
print("\n--- probe: is it a reasoning model, and does the incident prompt fit? ---")
probes = {}
for name in MODELS:
    try:
        ensure_ollama()
        p = probes[name] = probe_model.probe(name)
        print(f"\n{name}  (declared ctx {p['ctx']:,})")
        print(f"  reasoning model : {p['reasons']}")
        print(f"  incident prompt : est {p['est']:,} -> real {p['real']:,} (ratio {p['ratio']:.3f})")
        print(f"  truncated       : {p['truncated']}")
    except Exception as exc:
        print(f"\n{name}: probe failed — {exc}")
        tail = ollama_log_tail()
        if tail:
            print("  --- what the server actually said ---")
            print("  " + tail.replace("\n", "\n  "))
    finally:
        unload_model(name)

if len(probes) < len(MODELS):
    print(f"\nprobed {len(probes)} of {len(MODELS)} — missing: "
          + ", ".join(n for n in MODELS if n not in probes))

In [ ]:
# One model at a time, each with one retry behind a server restart.
#
# The matrix used to be a bare loop and it broke on Colab: after two or three
# models Ollama answers HTTP 500 on load while still serving /api/tags, because
# the previous models are still resident. Ollama holds a model for five minutes
# after its last request, so a four-model loop keeps two or three on the card at
# once.
#
# So each model is unloaded as soon as it is done — `keep_alive: 0`, Ollama's
# own mechanism. Restarting the server is kept as the fallback for when a run
# fails anyway. **Neither one deletes a model**: everything stays pulled on
# disk, because re-downloading gigabytes on conference wifi is not a recovery
# plan.
#
# A model that fails twice is RECORDED, not silently dropped. A matrix with a
# hole in it is a finding; a matrix that quietly has three rows instead of four
# is a lie you tell yourself later.
matrix, missing = {}, {}

for name, cfg in MODELS.items():
    slug = name.replace(":", "-")
    for attempt in (1, 2):
        try:
            ensure_ollama()
            matrix[name] = bench.run(MATRIX_ITEMS, model=name,
                                     out_path=f"bench-{MACHINE}-{slug}.json",
                                     progress=False, **cfg)
            unload_model(name)      # free the card before the next one loads
            break
        except Exception as exc:
            if attempt == 1:
                print(f"  {name}: {exc}\n    restarting the server and retrying once")
                tail = ollama_log_tail()
                if tail:
                    print("    --- what the server actually said ---")
                    print("    " + tail.replace("\n", "\n    "))
                restart_ollama()
            else:
                missing[name] = f"{type(exc).__name__}: {exc}"
                print(f"  {name}: failed twice — left out of the matrix")

print(f"\nran {len(matrix)} of {len(MODELS)} models")
if missing:
    print("--- missing, and why ---")
    for name, why in missing.items():
        print(f"  {name}: {why}")

In [ ]:
PRICE_IN, PRICE_OUT = measure.PRICE_IN, measure.PRICE_OUT

head = (f"{'model':16s} {'arch':9s} {'tok in':>8s} {'tok out':>8s} {'cut':>4s} "
        f"{'prec':>6s} {'exact':>6s} {'drift':>6s} {'$/1k runs':>10s} {'min':>6s}")
print(head)
print("-" * len(head))
for name, d in matrix.items():
    for arch, v in d["arch"].items():
        usd = (v["real_in"] * PRICE_IN + v["real_out"] * PRICE_OUT) / 1e6 * 1000
        print(f"{name:16s} {arch:9s} {v['real_in']:8,} {v['real_out']:8,} "
              f"{v.get('truncated', 0):4d} {v['precision']:6.0%} {v['exact']:6.0%} "
              f"{v.get('format_drift', 0):6d} {usd:10.2f} {v['seconds']/60:6.2f}")
    print()

cut = {n: sum(v.get("truncated", 0) for v in d["arch"].values()) for n, d in matrix.items()}
if any(cut.values()):
    print("!! TRUNCATED CALLS — those rows are not cost figures, they are the cost of")
    print("!! a prompt that was silently cut to fit. Cheap because work was thrown away:")
    for n, c in cut.items():
        if c:
            print(f"!!   {n}: {c} call(s)")

### What to look for

**Output tokens, not input.** Input is nearly identical across models — they
read the same prompt. Output is where they diverge, and output is the
expensive side. A model that thinks out loud costs multiples of one that
answers in fifteen tokens, for the same task.

**Cost against accuracy, not either alone.** Cheapest-and-wrong is not a
saving, and neither is the most accurate model if its answer cannot be parsed.
The `drift` column is there so a model cannot win on accuracy while quietly
breaking the contract it was given.

## 11. The eleven principles, against what this notebook actually measured

**Source:** Alex "Sandu" Astrum and Luke Schlangen, *Guide to AI Tokenomics:
Eleven Principles for Token Efficient Software Engineering*, Google Cloud
blog, 17 July 2026 —
<https://cloud.google.com/blog/topics/developers-practitioners/guide-to-ai-tokenomics-eleven-principles-for-token-efficient-software-engineering>

**The eleven titles below are quoted verbatim from that article, unchanged and
in its order.** Checked against the published page on 15 September 2026: all
eleven match character for character. Everything after each title — the
one-line restatement, the verdict, and the measurements — is this notebook's
own commentary, not Google's text.

It is a good list. It is also **entirely qualitative**: no measurement data, no
benchmarks, no tables, no charts, no before-and-after anywhere in it.

This notebook is the other half. Below, each principle gets the same three
things: what it asks for, what was measured here, and — where nothing was
measured — what it would take to measure it. **Four of the eleven are
demonstrated. The rest are not, and are marked as such.**

---

### 1. Start with a balanced model — **demonstrated**

Default to mid-tier, escalate only when a task actually fails.

Four models, same 15 items, same prompts. `qwen3:8b` — an 8B model — beat both
12B models on accuracy *and* cost: **40% exact at \$35.69** per 1000 runs
against 20% at \$39.78. Escalating to `deepseek-r1:8b` bought **73% exact for
\$125.92** — nearly double the accuracy at 3.5x the price.

That is the principle with a price tag on it: the escalation is worth it or it
is not, and now the question can be asked in numbers.

### 2. Use skills from the beginning — **not demonstrated**

Package recurring knowledge into reusable bundles (`SKILL.md`) instead of
re-explaining it every prompt.

The closest thing here is the opposite failure: the incident architecture
re-sends **13,466 tokens of category rules on every single call**. Moving them
out of the prompt — the `fetch_rules` node reading from storage — is the same
instinct, one layer down. To measure the principle properly you would need an
agent harness, not a classification workload.

### 3. Automate with scripts and CLI tools — **not demonstrated in the workload**

Build small local tools for repetitive chores.

The measurement harness *is* an instance of this — `probe_model.py`,
`bench.py`, `crossover.py` exist so the same work is not redone by hand — but
that is this notebook's own scaffolding, not something it measured.

### 4. Delegate output-heavy tasks — **not demonstrated, but priced**

Push verbose work into sub-agents so the main session reconciles results, not
whole trajectories.

Measured here: `deepseek-r1:8b` produced **7,737 output tokens** on the scoped
architecture and **18,890** on fan-out, where `qwen3:8b` needed 187. Output is
billed several times higher than input, so that verbosity is the entire cost
difference. A sub-agent returning only the id list is exactly this principle —
and the numbers above are what it would save.

### 5. Divide and conquer — **extendable, and the pieces are here**

Plan in one high-reasoning session, execute in a clean low-token one.

Both halves already exist in this notebook: an expensive accurate model and a
cheap fast one, with measured cost and accuracy for each. Routing the hard
items to `deepseek-r1:8b` and the rest to `qwen3:8b` is a small amount of code
on top of `run_live.py`, and the mixture's cost and accuracy can be compared
against either model alone. **This is the most valuable extension on the list**
because it is the only one that can be *run*, not just described.

### 6. Shift verification left — **demonstrated at the harness level**

Cheap checks early, expensive ones late.

`probe_model.py` answers three questions in four calls — is it a reasoning
model, does the prompt fit, how far is its tokenizer from the estimate —
before committing to a benchmark that takes twenty minutes. That habit was
learned the hard way here: a twenty-minute sweep was lost twice to a condition
a four-call probe would have caught.

### 7. Undo when adrift — **not applicable**

Revert rather than stacking corrective prompts. Written against Google's own
Antigravity UI, and there is no equivalent to measure in a batch workload.

### 8. Be specific with context — **demonstrated, and it is the whole talk**

Point at the exact section rather than issuing a broad request.

This is the incident. The agreed design sent the **2–5 categories a market
actually uses**; the system drifted into sending **all 50**. Measured on real
tokens: **\$429.22 against \$39.78 per 1000 runs — 10.8x**, for the same work,
the same model, the same answers.

Everything else on this list is worth single-digit percentages. This one is
worth an order of magnitude.

### 9. Iterate on rules — **not demonstrated**

Encode a repeated correction persistently (`AGENTS.md`) instead of re-prompting.

Adjacent, not the same: the taxonomy here *is* a persistent rules artefact, and
the category limit lived in code as a guardrail. What the incident shows is the
failure mode one step later — a rule that exists, is encoded, and is then
silently overridden by a change nobody read.

### 10. Avoid uncontrolled loops — **demonstrated twice, both by accident**

Give autonomous loops strict limits and stop conditions.

* The incident's retry ladder was **inverted**: 50 attempts in DEV, 30 in TST,
  10 in PRD. The environment with no spending limit had the most retries.
* Measured here: a reasoning model with **no stop token** consumed 256, then
  1,024, then **4,096 output tokens** and returned an empty answer every time.
  A loop with no stop condition, billed at output rates, producing nothing.

**Implemented in section 12** as a spending ceiling that stops the run — the
primitive none of the three agent SDKs provides.

### 11. Start new sessions for each new topic — **not demonstrated**

One chat per topic; start fresh when the topic changes.

The nearest measurement is the prefix cache: repeating a prompt hit **99.9%
cached**, switching to a different market hit **0%**, and returning to the
first after the switch recovered only **56%**. Topic switching has a price, and
that is what it looks like from the cache's side.

---

### What to take from the mapping

**Four demonstrated, one runnable extension, six not shown.** Said plainly
because a list of principles with no evidence is what we started from, and
replacing it with a list of claims and no evidence would be no better.

The asymmetry is the finding. Principle 8 — send only what the task needs — is
worth **10.8x** here. Principle 1 — pick the right model — is worth **10% of
cost and double the accuracy**. Everything else on the list is real, and small
by comparison. A list of eleven equal-looking bullet points hides that.

## 12. The two principles you can actually run

Nine of the eleven are habits. Two of them are code, and both are missing from
the tooling: **principle 5** (route cheap first, escalate on failure) and
**principle 10** (give the loop a stop condition).

Principle 10 is worth dwelling on. **None of the three agent SDKs surveyed for
this talk — ADK, LangGraph, OpenAI Agents — has a budget primitive.** All
three count tokens. None of them enforces money. In the incident behind this
notebook, that gap is the entire distance between a \$500 bill and a \$5,000
one.

What follows is about forty lines in `levers.py`.

### Principle 5 — route cheap first, escalate only on a visible failure

The escalation signal has to be readable **without knowing the right answer**,
or it is not a signal, it is hindsight. Two work here, and both were measured
above: the cheap model returning **nothing**, and the cheap model
**over-producing** — `qwen3:8b` hands back 2.7 categories per item where 1.3
are correct.

In [ ]:
import levers

ROUTE_ITEMS = 5          # raise once you know how slow the strong model is here
CHEAP, STRONG = "qwen3:8b", "deepseek-r1:8b"

routed = levers.run_routed(feedback.ITEMS[:ROUTE_ITEMS], CHEAP, STRONG, usd_limit=5.0)
print(f"\n  items      : {routed['items']}  ({routed['escalated']} escalated to {STRONG})")
print(f"  precision  : {routed['precision']:.0%}   exact: {routed['exact']:.0%}")
print(f"  tokens     : {routed['tokens_in']:,} in / {routed['tokens_out']:,} out")
print(f"  spent      : ${routed['usd']:.4f}")

### Principle 10 — a ceiling that stops the run

Set the limit below what the work costs and watch it refuse to continue. This
is the primitive the SDKs do not give you.

One honest limitation, visible in the output: the guard stops the **next**
call once the ceiling is passed. It cannot un-spend the call that crossed it,
so a run can overshoot by up to one call. A limit is not a guarantee.

In [ ]:
tiny = levers.run_routed(feedback.ITEMS[:ROUTE_ITEMS], CHEAP, STRONG,
                         usd_limit=0.005, progress=False)
print(f"  ceiling    : $0.0050")
print(f"  spent      : ${tiny['usd']:.4f}   after {tiny['items']} of {ROUTE_ITEMS} items")
print(f"  stopped    : {tiny['stopped']}")

### What the routing buys, against each model alone

Compare the mixture with the single-model rows from section 10. The question
a router answers is not "which model is best" but **"how much of the expensive
model do I actually need"** — and that is answerable only once both the cost
and the accuracy of each are measured.

The escalation rate is the number to watch. If almost nothing escalates, the
cheap model was enough and the strong one is a safety net. If almost
everything escalates, you have paid for two calls to get one answer, and the
router is costing you money rather than saving it.

### What it actually did here — and it is not an advertisement

Measured on all 15 items, `qwen3:8b` first, `deepseek-r1:8b` on failure:

| | cost per 1000 runs | exact match | accuracy per dollar |
|---|---|---|---|
| `qwen3:8b` alone | \$35.71 | 40% | **1.12 pp/\$** |
| **routed** | **\$58.80** | **47%** | 0.80 pp/\$ |
| `deepseek-r1:8b` alone | \$125.92 | 73% | 0.58 pp/\$ |

Three of fifteen items escalated. The router bought **7 percentage points for
65% more money** — real, and a worse rate than simply running the cheap model.
Escalation buys accuracy at a declining rate, and that is the honest shape of
this lever.

The reason is worth more than the table. `qwen3:8b` is wrong on 9 of 15 items;
the escalation signal — answered nothing, or over-produced — catches **3 of
those 9**. **A router is only as good as its ability to notice failure without
knowing the answer**, and that detector, not the switching, is the hard part.

Which is the same lesson as everything else here: the expensive mistake is not
picking the wrong model. It is not being able to tell that you did.

## 13. Citing this, and what it is built on

### Author

**Grzegorz Wasilewski**, Data & AI Senior Manager at Accenture —
[linkedin.com/in/legard77](https://www.linkedin.com/in/legard77)
Talk: *AI Tokenomics*, Code Europe, Warsaw, 15 September 2026.

### If you want to cite the notebook

```
Wasilewski, Grzegorz (2026). AI Tokenomics: measuring a prompt instead of guessing
at it. Notebook accompanying the talk at Code Europe, Warsaw,
15 September 2026. Accenture. https://github.com/acn-codeeurope/ai-tokenomics
```

BibTeX:

```bibtex
@misc{wasilewski2026tokenomics,
  author       = {Wasilewski, Grzegorz},
  title        = {AI Tokenomics: measuring a prompt instead of guessing at it},
  year         = {2026},
  month        = {9},
  howpublished = {Notebook accompanying the talk at Code Europe, Warsaw},
  organization = {Accenture},
  note         = {Presented 15 September 2026},
  url          = {https://github.com/acn-codeeurope/ai-tokenomics}
}
```

### What this is built on

The Colab bootstrap for Ollama — the `apt` prerequisites, the installer,
`Popen(['ollama','serve'])` and the readiness loop — is adapted from
[`Legard777/eskadra-bielik-misja2`](https://github.com/Legard777/eskadra-bielik-misja2)
(Apache-2.0), itself a fork of `avedave`'s work. See `NOTICE`.

The eleven principles in section 11 are Google Cloud's, titles quoted verbatim
and linked at the point of use. Big-T notation is Dan Neff's (Adobe), published
by the Tokenomics Foundation as a working draft, June 2026.

### Licence

Apache License 2.0. Use it, change it, build on it. Keep the attribution,
and say what you changed if you redistribute a modified version.

The measurements are reproducible: every number above comes from a run you can
repeat, and the result files from the runs behind this notebook ship alongside
it. Where a figure is calculated rather than measured, it is labelled as such
in the cell that produces it.

The views and the measurements here are the author's own.

### Primary sources this is built against

**Google Cloud** — *Guide to AI Tokenomics: Eleven Principles for Token
Efficient Software Engineering*, Alex Astrum and Luke Schlangen, 17 July 2026.
`cloud.google.com/blog/topics/developers-practitioners/guide-to-ai-tokenomics-eleven-principles-for-token-efficient-software-engineering`
The eleven principles mapped in section 11. Qualitative throughout: no
measurement data, no benchmarks, no tables.

**FinOps Foundation** — *Token Economics: The Atomic Unit of AI Value*,
J.R. Storment. `finops.org/insights/token-economics-the-atomic-unit-of-ai-value/`
Source of the definition of a token as a unit of computation rather than of
ownership. The FinOps Foundation is a project of the Linux Foundation.

**Tokenomics Foundation** — Big-T Notation, Dan Neff (Adobe), published by the
Tokenomics Foundation, working draft June 2026.
`tokeneconomics.com/docs/projects/big-t/big-t-notation-paper/`
The complexity ladder `T(1) … T(n·k·a) … T(∞)`. The Tokenomics Foundation is
hosted by the Linux Foundation. Note that the CC BY 4.0 notice on that site is
a site-footer licence, not a licence block on the paper, and the pages carry
no "how to cite" section.

**Ollama** — `docs.ollama.com`, and the `/api/generate` response fields
`prompt_eval_count`, `eval_count` and `prompt_eval_cached_count`, which are
what makes measurement possible here without a billing account.

**LangGraph** — `Send`, reducers and `add_conditional_edges`, used in
`graph_app.py` to make the fan-out, the join and the branch visible as named
primitives rather than hidden in a comprehension.

### What is not a source

The taxonomy, the feedback items and the ground-truth labels are synthetic and
were written for this notebook. The incident they are modelled on is real; its
data is not here, and no client is identifiable from anything in this repo.